# DVF data exploration
# Author: Xavier VAN AUSLOOS, 9/02/26
# Notebook for preparing dataset with all mutations in France for houses only
# DVF Data from 2020 to 2025

Load a DVF file from `data/raw/` and explore mutations (nature, date, value).

In [17]:
from pathlib import Path
import sys

# Ensure project src is on path (kernel cwd is often the notebook dir, not project root)
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
_src = _project_root / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from dvf import load_dvf_raw, load_dvf_csv, load_dvf_plus, summarize_mutations

In [18]:
# Data directory and list all files
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
raw_dir = _project_root / "data" / "raw"
all_files = sorted(raw_dir.glob("*"))
txt_files = sorted(raw_dir.glob("*.txt"))
print("All files in data/raw:", [f.name for f in all_files if not f.name.startswith(".")])
print(" .txt files (will be loaded):", [f.name for f in txt_files])

All files in data/raw: ['ValeursFoncieres-2020-S2.txt', 'ValeursFoncieres-2021.txt', 'ValeursFoncieres-2022.txt', 'ValeursFoncieres-2023.txt', 'ValeursFoncieres-2024.txt', 'ValeursFoncieres-2025-S1.txt', 'zip']
 .txt files (will be loaded): ['ValeursFoncieres-2020-S2.txt', 'ValeursFoncieres-2021.txt', 'ValeursFoncieres-2022.txt', 'ValeursFoncieres-2023.txt', 'ValeursFoncieres-2024.txt', 'ValeursFoncieres-2025-S1.txt']


In [ ]:
# Load all .txt files from raw (pipe-separated), concatenate into one DataFrame
import pandas as pd

if not txt_files:
    df = pd.DataFrame()
    print("No .txt files in data/raw.")
else:
    frames = []
    for p in txt_files:
        print(f"Loading {p.name}...")
        frames.append(load_dvf_raw(p, sep="|"))
    df = pd.concat(frames, ignore_index=True)
    print(f"Total rows: {len(df):,}")
df.head()

Loading ValeursFoncieres-2020-S2.txt...


In [ ]:
# filter df for houses only
df = df[df["Type local"] == "Maison"]
print("Total rows for houses mutations in France: ", len(df))



In [ ]:
# Row count and display all columns (no truncation)
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print("\nAll columns:")
display(df.columns.tolist())

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Preview: first 10 rows with all columns visible
display(df.head(10))

In [ ]:
# Filter: houses only, commune INSEE 13033 (Ensues)
df = df[(df["Code postal"] == 13820)]
df.head()
print("total rows: ", len(df))

In [ ]:
# Display all columns with their values (no truncation)
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", 100)  # increase if you need more rows
pd.set_option("display.max_colwidth", 40)  # max chars per column (None = no limit)
df

In [ ]:
# get min and max date (convert to datetime first; raw format is DD/MM/YYYY so string min/max was wrong)
date_col = "Date mutation"
date_series = pd.to_datetime(df[date_col], format="mixed", dayfirst=True, errors="coerce")
min_date = date_series.min()
max_date = date_series.max()
print(f"Mutation Date range: {min_date} to {max_date}")



In [ ]:
# Merge all rows related to same Code Commune + Prefixe de section + Section
# into a single row with all mutations as a JSON string
import json

# Group by these columns (identify unique properties)
# Note: Prefixe de section can be NaN - fill with empty string for consistent grouping
group_cols = ["Code commune", "Section", "No plan"]

# Columns to keep as house details (take from first row of each group)
house_cols = [
    "Code postal",
    "Commune",
    "Code departement",
    "Code commune",
    "Prefixe de section",
    "Section",
    "No plan",
    "Type local",
    "Surface reelle bati",
    "Surface terrain",
    "Nombre pieces principales",
]

# Group and aggregate
def create_mutations_json(group):
    """Build mutations JSON string from group rows."""
    mutations_list = [
        {row["Date mutation"]: str(row["Valeur fonciere"])}
        for _, row in group.iterrows()
    ]
    return json.dumps(mutations_list, ensure_ascii=False)

# Fill NaN in Prefixe de section with empty string for consistent grouping
# This ensures records with NaN Prefixe de section group together
df_grouping = df.copy()


# Group by Code commune + Prefixe de section + Section
grouped = df_grouping.groupby(group_cols, dropna=False)

# Build result rows
result_rows = []
for (code_commune, prefixe, section), group in grouped:
    house_info = group.iloc[0][house_cols].to_dict()
    house_info["mutations"] = create_mutations_json(group)
    result_rows.append(house_info)

df_grouped = pd.DataFrame(result_rows)
# Order df_grouped by Section
df_grouped = df_grouped.sort_values(by=["Code commune", "Section", "No plan"])

print(f"Original rows: {len(df)}")
print(f"Grouped rows: {len(df_grouped)}")

df_grouped


In [ ]:
# filter df_grouped for Section = "AT" and No plan = 113
#df_grouped = df_grouped[(df_grouped["Section"] == "AT") & (df_grouped["No plan"] == 113)]
#df_grouped

In [ ]:
# Save df_grouped to CSV in data/processed (project root)
from pathlib import Path

# Get project root (where notebooks folder is)
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent

# Ensure data/processed directory exists in project root
output_dir = _project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Save to CSV
output_file = output_dir / "df_grouped_ensues_2020_2025_houses.csv"
df_grouped.to_csv(output_file, index=False)
print(f"Saved to: {output_file.absolute()}")
print(f"File size: {output_file.stat().st_size / 1024:.2f} KB")
